# Simulación de impacto del semáforo de riesgo (datos sintéticos)
**Trabajo Final Integrador — Analítica de Negocios Avanzada (UCEMA, 2026)**

> ⚠️ **Este notebook es una SIMULACIÓN.** Los pasajeros, las conexiones, los costos de las acciones y su efectividad son **datos sintéticos** generados con supuestos documentados. Los resultados son estimaciones del impacto posible, **no resultados observados**. Los datos sintéticos se guardan aparte (`generated/sintetico/`, prefijo `sintetico_`) y nunca se mezclan con los datos reales.

**Objetivo (FASE 5):** estimar cuánto podría ahorrar la aerolínea si usa el semáforo de riesgo del modelo (`tp.ipynb`, sección 6.9) frente a gestionar las demoras recién cuando ocurren, calcular los KPIs estratégicos del canvas y estimar el **ingreso futuro en riesgo** por la pérdida de fidelidad de los pasajeros.

**Qué es real y qué es sintético:**

| Real (dataset de vuelos y modelo) | Sintético (supuestos) |
|---|---|
| Vuelos de test, si se retrasaron y cuántos minutos (`retraso_salida`) | Pasajeros por vuelo y pasajeros con conexión |
| Causa de cada demora (`retraso_por_*`), para saber si es controlable | Probabilidad de que el pasajero deje la aerolínea |
| Probabilidad y nivel del semáforo asignados por el modelo | Probabilidad de perder la conexión según la demora |
| Aerolínea y su modelo de negocio | Costo de cada acción y su efectividad |

`retraso_salida` y las causas de la demora no se pueden usar para predecir (leakage), pero sí para **medir el costo** de cada demora después de ocurrida.

**Período de las reglas:** las políticas de compensación son las vigentes en 2024-2025 (por ejemplo, el voucher de Southwest rige desde abril de 2024). La simulación responde cuánto ahorraría hoy el semáforo con las reglas actuales, aplicadas a una operación como la de 2022.

## 1. Configuración y supuestos
Todos los supuestos están en esta celda, con su fuente. Los marcados como **supuesto** no tienen una fuente directa: por eso los resultados se presentan con análisis de sensibilidad (sección 5).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
ARCHIVO_PREDICCIONES = 'generated/predicciones_test.csv'          # lo genera tp.ipynb (sección 6.9)
CARPETA_SINTETICO = 'generated/sintetico'
ARCHIVO_SINTETICO = f'{CARPETA_SINTETICO}/sintetico_pasajeros_vuelos_test.csv'

# --- Pasajeros (sintético, con base en fuentes) ---
ASIENTOS_POR_SEGMENTO = {'Regional': 76}   # supuesto: E175 / CRJ900 (76 asientos)
ASIENTOS_RESTO = 160                       # supuesto: A320 / 737 (aprox. 150-180 asientos)
OCUPACION = 0.815                          # BTS: load factor doméstico 2022
PROPORCION_CONEXION = 0.205                # DOT DB1C: 20,5% de los embarques domésticos son en conexión

# --- Conexiones perdidas (supuesto) ---
DEMORA_MEDIA_PERDIDA_MIN = 60   # demora con la que la mitad de los pasajeros con conexión la pierde
ESCALA_PERDIDA_MIN = 15         # qué tan rápido sube esa probabilidad alrededor de los 60 min

# --- Costos de la demora (con fuente) ---
COSTO_MINUTO_DEMORA = 100.76           # USD por minuto: A4A, costo de block time 2024 (tripulación, combustible, etc.)
COSTO_REUBICACION_REACTIVA = 250       # USD por pasajero: estimación de la industria (reubicación en EE. UU.)
VALOR_TIEMPO_PASAJERO_HORA = 47        # USD por hora: A4A, valor del tiempo del pasajero

# --- Compensación (solo demoras controlables; compromisos de las aerolíneas ante el DOT) ---
# En EE. UU. no hay compensación obligatoria por demoras (la propuesta del DOT se retiró en noviembre de 2025).
VOUCHER_COMIDA = 12                    # USD por pasajero, con esperas de 3 h o más (promedio reportado)
UMBRAL_COMIDA_MIN = 180
COSTO_HOTEL = 150                      # USD por pasajero: supuesto dentro de los topes reportados (USD 100-200)
UMBRAL_HOTEL_MIN = 360                 # supuesto: demoras de 6 h o más obligan a pasar la noche
SIN_COMPROMISO_COMIDA = ['G4']                 # Allegiant: sin compromisos de atención
SIN_COMPROMISO_HOTEL = ['G4', 'F9']            # Allegiant y Frontier: sin hotel
# Compensación propia por demoras controlables de 3 h o más (USD por pasajero)
COMPENSACION_FIJA = {'AS': 50, 'QX': 50,       # Alaska (y Horizon, que vuela como Alaska): código de USD 50
                     'HA': 50,                 # Hawaiian: supuesto igual a Alaska (monto no publicado)
                     'WN': 75}                 # Southwest: voucher de USD 75 (acuerdo con el DOT)
COMPENSACION_JETBLUE = [(360, 200), (300, 150), (240, 100), (180, 50)]   # JetBlue (B6): por horas de demora
UMBRAL_COMPENSACION_MIN = 180
# Causas que se consideran controlables por la aerolínea
CAUSAS_CONTROLABLES = ['retraso_por_aerolinea']   # avión tardío no se cuenta (criterio conservador)

# --- Fidelidad (ingreso futuro en riesgo; supuestos) ---
TARIFA_PROMEDIO = 378                  # USD: BTS, pasaje doméstico promedio 2022
VIAJES_POR_ANIO = 2                    # supuesto: viajes por año del cliente con la aerolínea (se varía de 1 a 4)
PROB_FUGA_POR_TRAMO = [(180, 0.06), (60, 0.03), (15, 0.01)]   # supuesto: prob. de dejar la aerolínea según la demora
REDUCCION_FUGA_AVISO = 0.10            # supuesto: el aviso anticipado reduce 10% esa probabilidad (se varía)

# --- Acciones del semáforo (supuestos) ---
COSTO_AVISO_PASAJERO = 0.10            # USD por pasajero avisado (notificación automática)
COSTO_REUBICACION_PREVENTIVA = 75      # USD por pasajero con conexión: ~30% de la reactiva (sin hotel ni comidas)
COSTO_REASIGNACION_VUELO = 1_500       # USD por vuelo rojo: tripulación de reserva y recursos
EFECTIVIDAD_BASE = 0.30                # % del daño que se evita al anticiparse (se varía en la sección 7)
UMBRAL_AMARILLO_SIM = 0.40             # cortes del semáforo de tp.ipynb (sección 6.9)
UMBRAL_ROJO_SIM = 0.60

| Supuesto | Valor | Fuente |
|---|---|---|
| Ocupación de los vuelos | 81,5% | BTS, *load factor* doméstico 2022 |
| Asientos por vuelo | 76 (regionales) / 160 (resto) | **Supuesto**: aviones típicos de cada segmento |
| Pasajeros con conexión | 20,5% | DOT DB1C (embarques domésticos en conexión) |
| Probabilidad de perder la conexión | 0 si la demora es ≤ 15 min; 50% con 60 min de demora | **Supuesto**: tiempos mínimos de conexión típicos de 45 a 60 min |
| Costo operativo por minuto de demora | USD 100,76 | Airlines for America (2024) |
| Compensación obligatoria | No existe para demoras domésticas | DOT (*Fly Rights*). La propuesta de compensación de 2024 se retiró en noviembre de 2025 (Federal Register) |
| ¿Demora controlable? | Si la causa principal registrada es la aerolínea | DOT: los compromisos solo cubren demoras controlables. Avión tardío no se cuenta (criterio conservador) |
| Comida | USD 12 por pasajero si la demora es ≥ 3 h (no Allegiant) | DOT dashboard (compromiso); monto promedio reportado |
| Hotel | USD 150 por pasajero si la demora es ≥ 6 h (no Allegiant ni Frontier) | DOT dashboard (compromiso); monto y umbral **supuestos** (topes reportados de USD 100-200) |
| Compensación propia (≥ 3 h) | JetBlue USD 50 a 200 según las horas; Alaska y Horizon USD 50; Southwest USD 75; Hawaiian USD 50 | Políticas de cada aerolínea y acuerdo DOT–Southwest; Hawaiian es **supuesto** |
| Valor del cliente | USD 378 por viaje × 2 viajes por año | BTS (tarifa 2022); viajes por año **supuesto** |
| Probabilidad de dejar la aerolínea | 1% (15-60 min), 3% (1-3 h), 6% (≥ 3 h) | **Supuesto** basado en Ferrer et al. (2012): las demoras reducen las compras futuras |
| Efecto del aviso sobre la fuga | Reduce 10% esa probabilidad | **Supuesto**: la evidencia (experimento de Via) indica que una disculpa sola tiene poco efecto |
| Reubicación reactiva (conexión perdida) | USD 250 por pasajero | Estimación de la industria del costo de reubicar pasajeros en EE. UU. |
| Valor del tiempo del pasajero | USD 47 por hora | Airlines for America |
| Aviso a pasajeros | USD 0,10 por pasajero | **Supuesto** |
| Reubicación preventiva | USD 75 por pasajero con conexión | **Supuesto** (más barata que la reactiva) |
| Reasignación de tripulación y recursos | USD 1.500 por vuelo | **Supuesto** |
| Efectividad de anticiparse | 30% (se varía de 0% a 100%) | **Supuesto clave**: ver análisis de sensibilidad |

## 2. Datos reales: predicciones del modelo en test
El archivo lo genera `tp.ipynb` al final de la sección 6.9: tiene los vuelos de test, su resultado real y el nivel del semáforo que les asignó el modelo elegido.

In [ ]:
pred = pd.read_csv(ARCHIVO_PREDICCIONES, parse_dates=['fecha_vuelo'])

print(f"Vuelos de test: {len(pred):,}")
print(f"Tasa de retraso: {pred['retrasado'].mean():.1%}")
print("\nVuelos por nivel del semáforo:")
display(pred['nivel_semaforo'].value_counts().reindex(['Verde', 'Amarillo', 'Rojo']).to_frame('cant_vuelos'))

# Demora controlable: la causa principal registrada es controlable por la aerolínea
causas = ['retraso_por_aerolinea', 'retraso_por_clima', 'retraso_por_trafico_aereo',
          'retraso_por_seguridad', 'retraso_por_avion_tardio']
tiene_causa = pred[causas].fillna(0).sum(axis=1) > 0
causa_principal = pred[causas].fillna(0).idxmax(axis=1).where(tiene_causa)
pred['controlable'] = causa_principal.isin(CAUSAS_CONTROLABLES)

retrasados = pred['retrasado'] == 1
print(f"\nVuelos retrasados con causa registrada: {tiene_causa[retrasados].mean():.1%}")
print(f"Vuelos retrasados con demora controlable: {pred.loc[retrasados, 'controlable'].mean():.1%}")
print("(Los retrasados sin causa registrada se consideran no controlables: criterio conservador.)")

## 3. Generación de los datos sintéticos
Para cada vuelo de test se generan sus pasajeros y cuántos tienen conexión, con semilla fija para que el resultado sea reproducible:
- **Pasajeros:** cada asiento está ocupado con probabilidad 81,5% (distribución binomial).
- **Pasajeros con conexión:** cada pasajero tiene conexión con probabilidad 20,5% (binomial).

La probabilidad de perder la conexión no se sortea: se usa su valor esperado, así los dos escenarios se comparan sobre exactamente los mismos pasajeros.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

sintetico = pred[['fecha_vuelo', 'aerolinea', 'codigo_aerolinea', 'origen', 'destino', 'modelo_negocio']].copy()
sintetico['asientos'] = pred['modelo_negocio'].map(ASIENTOS_POR_SEGMENTO).fillna(ASIENTOS_RESTO).astype(int)
sintetico['pasajeros'] = rng.binomial(sintetico['asientos'], OCUPACION)
sintetico['pasajeros_conexion'] = rng.binomial(sintetico['pasajeros'], PROPORCION_CONEXION)

os.makedirs(CARPETA_SINTETICO, exist_ok=True)
sintetico.to_csv(ARCHIVO_SINTETICO, index=False)
print(f"Archivo sintético guardado: {ARCHIVO_SINTETICO}")

display(sintetico[['asientos', 'pasajeros', 'pasajeros_conexion']].describe().round(1))

## 4. Modelo de costos
Se comparan dos escenarios sobre los mismos vuelos de test:

**Sin modelo:** las demoras se gestionan cuando ocurren. Cada vuelo retrasado genera:
- **costo operativo:** minutos de demora × USD 100,76;
- **compensación (solo demoras controlables):** comida (≥ 3 h), hotel (≥ 6 h) y la compensación propia de cada aerolínea (≥ 3 h);
- **reubicación reactiva:** pasajeros que pierden la conexión × USD 250.

Además, **todas** las demoras (también las de clima) ponen en riesgo la fidelidad del pasajero: se calcula como un indicador aparte, el **ingreso futuro en riesgo** (sección 6).

**Con semáforo:** se aplican las acciones según el nivel. Las acciones **se pagan en todos los vuelos alertados**, también en las falsas alarmas:

| Nivel | Acción | Costo | Beneficio (si el vuelo efectivamente se retrasa) |
|---|---|---|---|
| 🟡 Amarillo y 🔴 Rojo | Aviso a los pasajeros | USD 0,10 por pasajero | Menos tiempo de espera y menos riesgo de perder al cliente (sección 6) |
| 🔴 Rojo | Reasignación de tripulación y recursos | USD 1.500 por vuelo | La demora se reduce según la efectividad: baja el costo operativo, la compensación y las conexiones perdidas |
| 🔴 Rojo | Reubicación preventiva de pasajeros con conexión | USD 75 por pasajero con conexión | Se evita la reubicación reactiva según la efectividad |

In [ ]:
def prob_perder_conexion(demora_min):
    # Supuesto: 0 si la demora es <= 15 min; curva logística con 50% de probabilidad a los 60 min
    p = 1 / (1 + np.exp(-(demora_min - DEMORA_MEDIA_PERDIDA_MIN) / ESCALA_PERDIDA_MIN))
    return np.where(demora_min > 15, p, 0.0)


def compensacion_por_pasajero(demora_min):
    # USD por pasajero que paga la aerolínea según la demora, su política y si la demora es controlable
    codigo = pred['codigo_aerolinea'].to_numpy()
    comida = np.where(~np.isin(codigo, SIN_COMPROMISO_COMIDA) & (demora_min >= UMBRAL_COMIDA_MIN), VOUCHER_COMIDA, 0)
    hotel = np.where(~np.isin(codigo, SIN_COMPROMISO_HOTEL) & (demora_min >= UMBRAL_HOTEL_MIN), COSTO_HOTEL, 0)
    propia = np.where(demora_min >= UMBRAL_COMPENSACION_MIN, pd.Series(codigo).map(COMPENSACION_FIJA).fillna(0), 0)
    jetblue = np.select([demora_min >= m for m, _ in COMPENSACION_JETBLUE], [v for _, v in COMPENSACION_JETBLUE], 0)
    propia = np.where(codigo == 'B6', jetblue, propia)
    return np.where(pred['controlable'].to_numpy(), comida + hotel + propia, 0)


def prob_fuga(demora_min):
    # Supuesto: probabilidad de que el pasajero deje de volar con la aerolínea según la demora (cualquier causa)
    return np.select([demora_min >= m for m, _ in PROB_FUGA_POR_TRAMO], [p for _, p in PROB_FUGA_POR_TRAMO], 0.0)


def costos_demora(demora_min, pasajeros, pasajeros_conexion):
    # Costo operativo, compensación y conexiones perdidas (valor esperado) de una demora
    operativo = demora_min * COSTO_MINUTO_DEMORA
    compensacion = pasajeros * compensacion_por_pasajero(demora_min)
    conexiones_perdidas = pasajeros_conexion * prob_perder_conexion(demora_min)
    return operativo, compensacion, conexiones_perdidas


def simular(efectividad=EFECTIVIDAD_BASE, factor_costo_acciones=1.0,
            reduccion_fuga=REDUCCION_FUGA_AVISO, viajes_por_anio=VIAJES_POR_ANIO):
    # Costos por vuelo en los dos escenarios. factor_costo_acciones escala el costo de las tres acciones.
    pax = sintetico['pasajeros'].to_numpy()
    pax_con = sintetico['pasajeros_conexion'].to_numpy()
    # Solo los vuelos retrasados (> 15 min) generan costos de demora
    demora = np.where(pred['retrasado'] == 1, pred['retraso_salida'].clip(lower=0), 0.0)
    rojo = (pred['nivel_semaforo'] == 'Rojo').to_numpy()
    alerta = pred['nivel_semaforo'].isin(['Amarillo', 'Rojo']).to_numpy()

    # Escenario sin modelo
    op, comp, perdidas = costos_demora(demora, pax, pax_con)
    reub = perdidas * COSTO_REUBICACION_REACTIVA

    # Escenario con semáforo: en rojo, la reasignación reduce la demora según la efectividad
    demora_sem = np.where(rojo, demora * (1 - efectividad), demora)
    op_s, comp_s, perdidas_s = costos_demora(demora_sem, pax, pax_con)
    # En rojo, la reubicación preventiva evita parte de las conexiones perdidas
    perdidas_s = np.where(rojo, perdidas_s * (1 - efectividad), perdidas_s)
    reub_s = perdidas_s * COSTO_REUBICACION_REACTIVA

    aviso = alerta * pax * COSTO_AVISO_PASAJERO * factor_costo_acciones
    reasignacion = rojo * COSTO_REASIGNACION_VUELO * factor_costo_acciones
    reub_preventiva = rojo * pax_con * COSTO_REUBICACION_PREVENTIVA * factor_costo_acciones

    # Impacto en el pasajero: horas de espera (el aviso permite evitar parte de la espera)
    horas = pax * demora / 60
    horas_s = np.where(alerta, pax * demora_sem / 60 * (1 - efectividad), pax * demora_sem / 60)

    # Fidelidad: ingreso futuro en riesgo (demoras por cualquier causa). El aviso reduce la probabilidad de fuga.
    valor_cliente = TARIFA_PROMEDIO * viajes_por_anio
    fuga = pax * prob_fuga(demora) * valor_cliente
    fuga_s = pax * prob_fuga(demora_sem) * np.where(alerta, 1 - reduccion_fuga, 1) * valor_cliente

    return pd.DataFrame({
        'retrasado': pred['retrasado'].to_numpy(), 'nivel': pred['nivel_semaforo'].to_numpy(),
        'pasajeros_conexion': pax_con,
        'sin_operativo': op, 'sin_compensacion': comp, 'sin_reubicacion': reub, 'sin_conexiones_perdidas': perdidas,
        'con_operativo': op_s, 'con_compensacion': comp_s, 'con_reubicacion': reub_s,
        'con_conexiones_perdidas': perdidas_s,
        'con_aviso': aviso, 'con_reasignacion': reasignacion, 'con_reubicacion_preventiva': reub_preventiva,
        'sin_horas_espera': horas, 'con_horas_espera': horas_s,
        'sin_ingreso_en_riesgo': fuga, 'con_ingreso_en_riesgo': fuga_s,
    })


def resumen_escenarios(sim):
    # Costos totales (USD) de cada escenario, por rubro
    filas = {
        'Costo operativo de la demora': (sim['sin_operativo'].sum(), sim['con_operativo'].sum()),
        'Compensación (comida, hotel y propia)': (sim['sin_compensacion'].sum(), sim['con_compensacion'].sum()),
        'Reubicación reactiva': (sim['sin_reubicacion'].sum(), sim['con_reubicacion'].sum()),
        'Acción: aviso a pasajeros': (0, sim['con_aviso'].sum()),
        'Acción: reasignación de recursos': (0, sim['con_reasignacion'].sum()),
        'Acción: reubicación preventiva': (0, sim['con_reubicacion_preventiva'].sum()),
    }
    tabla = pd.DataFrame(filas, index=['Sin modelo', 'Con semáforo']).T
    tabla.loc['TOTAL'] = tabla.sum()
    tabla['Diferencia'] = tabla['Sin modelo'] - tabla['Con semáforo']
    return tabla

## 5. Resultados del escenario base (efectividad 30%)

In [ ]:
sim_base = simular()
tabla_base = resumen_escenarios(sim_base)
display((tabla_base / 1e6).round(2).rename(columns=lambda c: f'{c} (millones de USD)'))

ahorro_base = tabla_base.loc['TOTAL', 'Diferencia']
print(f"Ahorro neto del semáforo (escenario base): USD {ahorro_base:,.0f}")
print(f"Ahorro neto por vuelo de test: USD {ahorro_base / len(sim_base):,.1f}")

### KPIs estratégicos del canvas
Se calculan con las definiciones del canvas. En el escenario con semáforo, la reubicación incluye la preventiva y la reactiva.

In [ ]:
demorados = (sim_base['retrasado'] == 1).sum()
rojo = sim_base['nivel'] == 'Rojo'
reubicados_sin = sim_base['sin_conexiones_perdidas'].sum()
reubicados_con = sim_base['con_conexiones_perdidas'].sum() + sim_base.loc[rojo, 'pasajeros_conexion'].sum()

kpis = pd.DataFrame({
    'Sin modelo': [
        sim_base['sin_compensacion'].sum() / demorados,
        sim_base['sin_reubicacion'].sum() / reubicados_sin,
        sim_base['sin_operativo'].sum() / demorados,
    ],
    'Con semáforo': [
        sim_base['con_compensacion'].sum() / demorados,
        (sim_base['con_reubicacion'].sum() + sim_base['con_reubicacion_preventiva'].sum()) / reubicados_con,
        sim_base['con_operativo'].sum() / demorados,
    ],
}, index=['Costo de compensaciones por vuelo demorado (USD)',
          'Costo de reubicación por pasajero reubicado (USD)',
          'Costo operativo por vuelo demorado (USD)'])
display(kpis.round(1))

## 6. Impacto en el pasajero y en la fidelidad
Dos indicadores que **no se suman al ahorro operativo**, porque no son costos directos de la aerolínea:
- **Horas de espera:** el aviso anticipado le permite al pasajero reorganizarse y esperar menos. Se valora con el valor del tiempo de A4A (USD 47 por hora).
- **Ingreso futuro en riesgo:** las demoras, también las de clima, reducen las compras futuras del pasajero (Ferrer et al., 2012). Se estima como pasajeros afectados × probabilidad de dejar la aerolínea × valor anual del cliente. Es la parte con más supuestos del modelo: por eso se muestra con su propia sensibilidad.

In [ ]:
horas_sin = sim_base['sin_horas_espera'].sum()
horas_con = sim_base['con_horas_espera'].sum()
print(f"Horas de espera de los pasajeros sin modelo:   {horas_sin:,.0f}")
print(f"Horas de espera de los pasajeros con semáforo: {horas_con:,.0f}")
print(f"Horas evitadas: {horas_sin - horas_con:,.0f} "
      f"(valor del tiempo: USD {(horas_sin - horas_con) * VALOR_TIEMPO_PASAJERO_HORA:,.0f})")

riesgo_sin = sim_base['sin_ingreso_en_riesgo'].sum()
riesgo_con = sim_base['con_ingreso_en_riesgo'].sum()
print(f"\nIngreso futuro en riesgo sin modelo:   USD {riesgo_sin:,.0f}")
print(f"Ingreso futuro en riesgo con semáforo: USD {riesgo_con:,.0f}")
print(f"Ingreso protegido por el semáforo:     USD {riesgo_sin - riesgo_con:,.0f}")

**Sensibilidad del ingreso protegido:** según cuánto reduzca el aviso la probabilidad de fuga y cuántos viajes por año haga el cliente con la aerolínea.

In [ ]:
reducciones = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]
viajes = [1, 2, 3, 4]
tabla_fidelidad = pd.DataFrame({
    f'{v} viaje(s) por año': [
        (lambda sim: (sim['sin_ingreso_en_riesgo'].sum() - sim['con_ingreso_en_riesgo'].sum()) / 1e6)(
            simular(reduccion_fuga=r, viajes_por_anio=v))
        for r in reducciones]
    for v in viajes
}, index=[f'Aviso reduce la fuga {r:.0%}' for r in reducciones])
print("Ingreso futuro protegido por el semáforo (millones de USD):")
display(tabla_fidelidad.round(2))

## 7. Análisis de sensibilidad
La efectividad de anticiparse y el costo de las acciones no tienen una fuente directa. En lugar de apostar a un valor, se prueba un rango y se muestra **a partir de qué efectividad el semáforo se paga solo** (punto de equilibrio), con tres niveles de costo de las acciones: la mitad, el base y el doble.

In [ ]:
efectividades = np.linspace(0, 1, 21)
factores = {'Acciones a mitad de costo': 0.5, 'Costo base de las acciones': 1.0, 'Acciones al doble de costo': 2.0}

sensibilidad = pd.DataFrame({
    nombre: [resumen_escenarios(simular(e, f)).loc['TOTAL', 'Diferencia'] / 1e6 for e in efectividades]
    for nombre, f in factores.items()
}, index=efectividades)

def punto_equilibrio(serie):
    # Menor efectividad con ahorro neto positivo (interpolación lineal entre los puntos probados)
    positivos = serie[serie > 0]
    if positivos.empty:
        return np.nan
    e1 = positivos.index[0]
    if e1 == serie.index[0]:
        return e1
    e0 = serie.index[serie.index.get_loc(e1) - 1]
    return e0 + (0 - serie[e0]) * (e1 - e0) / (serie[e1] - serie[e0])

equilibrios = sensibilidad.apply(punto_equilibrio)

fig, ax = plt.subplots(figsize=(10, 5.5))
for nombre, color in zip(factores, ['seagreen', 'steelblue', 'darkorange']):
    ax.plot(efectividades * 100, sensibilidad[nombre], marker='o', markersize=3, color=color,
            label=f"{nombre} (equilibrio: {equilibrios[nombre]:.0%})" if pd.notna(equilibrios[nombre])
            else f"{nombre} (sin equilibrio)")
ax.axhline(0, color='black', linewidth=1)
ax.axvline(EFECTIVIDAD_BASE * 100, color='gray', linestyle='--', label=f'Escenario base ({EFECTIVIDAD_BASE:.0%})')
ax.set_title('Ahorro neto del semáforo según la efectividad de anticiparse (vuelos de test)')
ax.set_xlabel('Efectividad: % del daño que se evita al anticiparse')
ax.set_ylabel('Ahorro neto (millones de USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display((equilibrios * 100).round(0).to_frame('Efectividad mínima para que convenga (%)'))

## 8. Comparación de políticas de acción (validación)
La simulación del escenario base mostró que aplicar acciones caras a todos los vuelos rojos no se paga sola. Acá se comparan **políticas alternativas**: qué acción se toma desde qué probabilidad.

**Para no elegir con el test**, las políticas se comparan con las predicciones de **validación** (vuelos de train predichos con validación cruzada, `generated/predicciones_validacion.csv`). La política elegida se mide en test **una sola vez**, al final. Los supuestos de costos y efectividad son los mismos de la sección 1: no se cambian para favorecer a ninguna política.

| Política | Aviso | Reubicación preventiva | Reasignación de recursos |
|---|---|---|---|
| Actual (3 niveles) | ≥ 0,40 | ≥ 0,60 | ≥ 0,60 |
| Solo aviso | ≥ 0,40 | — | — |
| 4 niveles (corte X) | ≥ 0,40 | — | ≥ X (X de 0,70 a 0,90) |

En 4 niveles, el nivel naranja (0,60 a X) recibe aviso y monitoreo prioritario de operaciones, sin costo directo. La reubicación preventiva no se incluye porque, con estos supuestos, no se paga en ningún nivel: reubica a todos los pasajeros con conexión, y la mayoría no la habría perdido.

Los resultados se comparan **por vuelo** (USD por vuelo), porque validación y test tienen distinta cantidad de vuelos.

In [ ]:
ARCHIVO_PREDICCIONES_VAL = 'generated/predicciones_validacion.csv'
ARCHIVO_SINTETICO_VAL = f'{CARPETA_SINTETICO}/sintetico_pasajeros_vuelos_validacion.csv'


def preparar_datos(archivo, archivo_sintetico):
    # Carga predicciones, marca demoras controlables y genera pasajeros sintéticos (semilla fija)
    d = pd.read_csv(archivo, parse_dates=['fecha_vuelo'])
    tiene_causa = d[causas].fillna(0).sum(axis=1) > 0
    d['controlable'] = d[causas].fillna(0).idxmax(axis=1).where(tiene_causa).isin(CAUSAS_CONTROLABLES)
    rng_d = np.random.default_rng(RANDOM_STATE)
    asientos = d['modelo_negocio'].map(ASIENTOS_POR_SEGMENTO).fillna(ASIENTOS_RESTO).astype(int)
    d['pasajeros'] = rng_d.binomial(asientos, OCUPACION)
    d['pasajeros_conexion'] = rng_d.binomial(d['pasajeros'], PROPORCION_CONEXION)
    d[['fecha_vuelo', 'codigo_aerolinea', 'origen', 'destino', 'pasajeros', 'pasajeros_conexion']].to_csv(
        archivo_sintetico, index=False)
    return d


def compensacion_datos(d, demora_min):
    # Misma regla de compensación de la sección 4, para cualquier conjunto de vuelos
    codigo = d['codigo_aerolinea'].to_numpy()
    comida = np.where(~np.isin(codigo, SIN_COMPROMISO_COMIDA) & (demora_min >= UMBRAL_COMIDA_MIN), VOUCHER_COMIDA, 0)
    hotel = np.where(~np.isin(codigo, SIN_COMPROMISO_HOTEL) & (demora_min >= UMBRAL_HOTEL_MIN), COSTO_HOTEL, 0)
    propia = np.where(demora_min >= UMBRAL_COMPENSACION_MIN, pd.Series(codigo).map(COMPENSACION_FIJA).fillna(0), 0)
    jetblue = np.select([demora_min >= m for m, _ in COMPENSACION_JETBLUE], [v for _, v in COMPENSACION_JETBLUE], 0)
    propia = np.where(codigo == 'B6', jetblue, propia)
    return d['pasajeros'].to_numpy() * np.where(d['controlable'].to_numpy(), comida + hotel + propia, 0)


def simular_politica(d, umbral_aviso=None, umbral_reubicacion=None, umbral_reasignacion=None,
                     efectividad=EFECTIVIDAD_BASE):
    # Costos de una política de acción: cada acción se aplica a los vuelos con probabilidad >= su umbral
    proba = d['proba_retraso'].to_numpy()
    pax, pax_con = d['pasajeros'].to_numpy(), d['pasajeros_conexion'].to_numpy()
    retrasado = d['retrasado'].to_numpy() == 1
    demora = np.where(retrasado, d['retraso_salida'].clip(lower=0), 0.0)
    activa = lambda u: (proba >= u) if u is not None else np.zeros(len(d), dtype=bool)
    aviso, reub, reasig = activa(umbral_aviso), activa(umbral_reubicacion), activa(umbral_reasignacion)

    def costo_demora(dem, perdidas_factor):
        perdidas = pax_con * prob_perder_conexion(dem) * perdidas_factor
        return dem * COSTO_MINUTO_DEMORA + compensacion_datos(d, dem) + perdidas * COSTO_REUBICACION_REACTIVA

    sin = costo_demora(demora, 1.0)
    demora_p = np.where(reasig, demora * (1 - efectividad), demora)
    con = costo_demora(demora_p, np.where(reub, 1 - efectividad, 1.0))
    acciones = (aviso * pax * COSTO_AVISO_PASAJERO + reasig * COSTO_REASIGNACION_VUELO
                + reub * pax_con * COSTO_REUBICACION_PREVENTIVA)

    valor_cliente = TARIFA_PROMEDIO * VIAJES_POR_ANIO
    fuga = pax * prob_fuga(demora) * valor_cliente
    fuga_p = pax * prob_fuga(demora_p) * np.where(aviso, 1 - REDUCCION_FUGA_AVISO, 1) * valor_cliente
    horas = pax * demora / 60
    horas_p = np.where(aviso, pax * demora_p / 60 * (1 - efectividad), pax * demora_p / 60)

    n = len(d)
    return {
        '% vuelos con reasignación': reasig.mean() * 100,
        'Precision reasignación (%)': retrasado[reasig].mean() * 100 if reasig.any() else np.nan,
        'Ahorro en demoras (USD/vuelo)': (sin - con).sum() / n,
        'Costo de acciones (USD/vuelo)': acciones.sum() / n,
        'Ahorro neto operativo (USD/vuelo)': (sin - con - acciones).sum() / n,
        'Ingreso protegido (USD/vuelo)': (fuga - fuga_p).sum() / n,
        'Horas de espera evitadas (por vuelo)': (horas - horas_p).sum() / n,
    }


politicas = {'Actual (3 niveles)': dict(umbral_aviso=UMBRAL_AMARILLO_SIM, umbral_reubicacion=UMBRAL_ROJO_SIM,
                                        umbral_reasignacion=UMBRAL_ROJO_SIM),
             'Solo aviso': dict(umbral_aviso=UMBRAL_AMARILLO_SIM)}
for corte in [0.70, 0.75, 0.80, 0.85, 0.90]:
    politicas[f'4 niveles (reasignación >= {corte:.2f})'] = dict(umbral_aviso=UMBRAL_AMARILLO_SIM,
                                                                 umbral_reasignacion=corte)

In [ ]:
datos_val = preparar_datos(ARCHIVO_PREDICCIONES_VAL, ARCHIVO_SINTETICO_VAL)
print(f"Vuelos de validación: {len(datos_val):,} (tasa de retraso {datos_val['retrasado'].mean():.1%})")

comparacion_val = pd.DataFrame({nombre: simular_politica(datos_val, **p) for nombre, p in politicas.items()}).T
display(comparacion_val.round(2).sort_values('Ahorro neto operativo (USD/vuelo)', ascending=False))

**Elección de la política:** la de mayor **ahorro neto operativo por vuelo en validación**. El ingreso protegido y las horas evitadas se miran como información complementaria (impacto en el cliente), pero no deciden, porque tienen supuestos más fuertes. Completar el nombre en la celda siguiente después de ver la tabla.

In [ ]:
# Completar con la política elegida en validación, tal como aparece en la tabla
politica_elegida = None
assert politica_elegida in politicas, f'Elegí una política de esta lista: {list(politicas)}'

# Medición en test, UNA sola vez, con la política elegida
datos_test = preparar_datos(ARCHIVO_PREDICCIONES, ARCHIVO_SINTETICO)
resultado_test = pd.DataFrame({
    f'{politica_elegida} (test)': simular_politica(datos_test, **politicas[politica_elegida]),
    'Actual (3 niveles) (test, referencia)': simular_politica(datos_test, **politicas['Actual (3 niveles)']),
}).T
display(resultado_test.round(2))
print(f"Ahorro neto operativo total en test: "
      f"USD {resultado_test.iloc[0]['Ahorro neto operativo (USD/vuelo)'] * len(datos_test):,.0f}")

## 9. Conclusiones y limitaciones
> ⚠️ Completar con los resultados después de correr el notebook.

**Limitaciones:**
- Pasajeros, conexiones, costos de las acciones y efectividad son **sintéticos**: los resultados dependen de los supuestos, por eso se presentan con análisis de sensibilidad.
- Se simula sobre la muestra de test (25% de los vuelos de la muestra del 25% de 2022), así que los montos totales no representan a toda la operación: conviene leerlos **por vuelo** o en términos relativos.
- No se incluyen los vuelos cancelados ni desviados, que se excluyeron de la ABT.
- En EE. UU. no hay compensación obligatoria por demoras: se modelan los compromisos voluntarios de las aerolíneas (vigentes en 2024-2025) aplicados a vuelos de 2022. La propuesta del DOT de 2024 (USD 200–300 por demoras de 3 a 6 h) se retiró en noviembre de 2025.
- Los vouchers y créditos se valoran a su valor nominal, aunque a la aerolínea le cuestan menos (muchos no se usan): el costo de compensación está sobreestimado.
- Solo se cuenta como controlable la causa "aerolínea". Avión tardío puede ser controlable o no según qué demoró al vuelo anterior; contarlo como no controlable es un criterio conservador.
- El ingreso futuro en riesgo es un indicador aparte, con supuestos fuertes (probabilidad de fuga y efecto del aviso), y no se suma al ahorro operativo.
- La efectividad se aplica igual a todos los vuelos rojos. En la práctica, operaciones decide caso por caso (*human in the loop*).